# Identify which chain matches a protein sequence

Give this notebook a structure file (`.pdb`, `.cif`, `.mmcif`, optionally `.gz`) and a protein sequence or subsequence. It aligns the query to every polymer chain (local alignment, so a fragment is fine) and reports which chain(s) match, with identity, query coverage and the matching residue numbers.

Notes:
- Chain sequences are read from the **resolved residues** in the coordinates (ATOM/HETATM), so unresolved residues are absent and show up as gaps/shorter coverage.
- Identical copies of a protein (e.g. A and D in a dimer) will tie. All tied chains are listed.

In [ ]:
# Run once if needed
# %pip install biopython pandas

In [6]:
# ---- INPUTS: edit these ----
STRUCTURE_FILE = '1G8-PTRAMP-CSS.pdb'   # path to .pdb / .cif / .mmcif (.gz ok)

# Full sequence or a subsequence. FASTA header lines, spaces and newlines are ignored.
QUERY = '''
MIDVLLNKTNYLYFTFLLFLYISSNDLCSCNLKVRNNWEHFLQVYQNNGNFKMEILEPEC
TIQKNLPSEEEKEEFIDFDRKELLLHDFVVIKNSQFTDEKTLELYLSKEENNKSYLMLTF
YLGDLKLMIGHNSPYEISLILNVATVNKNMSNCKNNSYNIVLLKTTDVFNTSDLEILEGP
IQFSLGKSSGAFRINVTNFFLNNTWKAFTKDKISFLIKPEENCYTILENKINKPQLLIEK
KTTFYSEWGEWSNCSMDCDHPDNVQIRERECIHPSGDCFKGDLKESRPCIIPLPPCNELF
SHKDNSTFKILMIILPIVLVISIITILYHIFYKRKGAEKELYENVAGRYMYD
'''

# Thresholds for calling a chain a confident match
MIN_IDENTITY = 0.90   # identical / aligned positions
MIN_COVERAGE = 0.80   # fraction of the query that aligns to the chain

In [7]:
import gzip, io, os, re, warnings
import pandas as pd
from Bio import Align
from Bio.Align import substitution_matrices
from Bio.PDB import PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.PDBExceptions import PDBConstructionWarning
from Bio.SeqUtils import seq1

warnings.simplefilter('ignore', PDBConstructionWarning)

def clean_query(text):
    lines = [l for l in text.strip().splitlines() if not l.startswith('>')]
    seq = re.sub(r'[^A-Za-z]', '', ''.join(lines)).upper()
    if not seq or 'PASTE_SEQUENCE_HERE' in text:
        raise ValueError('Set QUERY to a protein sequence.')
    return seq

def load_structure(path):
    name = os.path.basename(path).lower()
    opener = gzip.open if name.endswith('.gz') else open
    base = name[:-3] if name.endswith('.gz') else name
    parser = MMCIFParser(QUIET=True) if base.endswith(('.cif', '.mmcif')) else PDBParser(QUIET=True)
    with opener(path, 'rt') as fh:
        return parser.get_structure('s', io.StringIO(fh.read()))

def chain_sequences(structure):
    '''Return {chain_id: (sequence, [residue labels])} for protein chains of model 0.'''
    out = {}
    for chain in structure[0]:
        seq, labels = [], []
        for res in chain:
            if res.id[0] == 'W':
                continue
            if not (is_aa(res, standard=False) or res.get_resname() == 'MSE'):
                continue
            seq.append(seq1(res.get_resname(), custom_map={'MSE': 'M'}, undef_code='X'))
            labels.append(f'{res.id[1]}{res.id[2].strip()}')
        if seq:
            out[chain.id] = (''.join(seq), labels)
    return out

def make_aligner():
    a = Align.PairwiseAligner()
    a.mode = 'local'
    a.substitution_matrix = substitution_matrices.load('BLOSUM62')
    a.open_gap_score = -10
    a.extend_gap_score = -0.5
    return a

def score_chain(aligner, query, chain_seq, labels):
    aln = aligner.align(chain_seq, query)[0]   # target = chain, query = query
    matches = aligned = 0
    mapped = []
    for (t0, t1), (q0, q1) in zip(*aln.aligned):
        for t, q in zip(range(t0, t1), range(q0, q1)):
            aligned += 1
            if chain_seq[t] == query[q]:
                matches += 1
            mapped.append(t)
    return dict(
        score=aln.score,
        identity=matches / aligned if aligned else 0.0,
        coverage=aligned / len(query),
        matches=matches,
        chain_len=len(chain_seq),
        first_res=labels[mapped[0]] if mapped else None,
        last_res=labels[mapped[-1]] if mapped else None,
    ), aln

In [8]:
query = clean_query(QUERY)
structure = load_structure(STRUCTURE_FILE)
chains = chain_sequences(structure)
print(f'Query length: {len(query)}')
print(f'Protein chains found: {", ".join(f"{c} ({len(s)} aa)" for c, (s, _) in chains.items())}')

aligner = make_aligner()
rows, alns = [], {}
for cid, (seq, labels) in chains.items():
    stats, aln = score_chain(aligner, query, seq, labels)
    rows.append(dict(chain=cid, **stats))
    alns[cid] = aln

df = (pd.DataFrame(rows)
      .sort_values(['matches', 'score'], ascending=False)
      .reset_index(drop=True))
df['confident'] = (df.identity >= MIN_IDENTITY) & (df.coverage >= MIN_COVERAGE)
df.style.format({'score': '{:.1f}', 'identity': '{:.1%}', 'coverage': '{:.1%}'})

Query length: 352
Protein chains found: D (264 aa), C (124 aa), A (267 aa), F (124 aa), E (21 aa), B (21 aa)


,chain,score,identity,coverage,matches,chain_len,first_res,last_res,confident
0,A,86.5,36.5%,51.4%,66,267,26,240,False
1,D,84.5,36.4%,50.0%,64,264,34,240,False
2,E,113.0,100.0%,6.0%,21,21,42,62,False
3,B,113.0,100.0%,6.0%,21,21,42,62,False
4,C,35.0,29.4%,9.7%,10,124,82C,115,False
5,F,35.0,29.4%,9.7%,10,124,82C,115,False


In [9]:
top = df.iloc[0]
tied = df[(df.matches == top.matches) & (df.identity == top.identity)]
tied_ids = tied.chain.tolist()

if not top.confident:
    print(f'No confident match (best: chain {top.chain}, identity {top.identity:.1%}, coverage {top.coverage:.1%}).')
    print('Check the sequence, lower MIN_IDENTITY/MIN_COVERAGE, or note that the protein may be absent from this file.')
else:
    print(f'Matching chain(s): {", ".join(tied_ids)}')
    for _, r in tied.iterrows():
        print(f'  chain {r.chain}: identity {r.identity:.1%}, query coverage {r.coverage:.1%}, '
              f'residues {r.first_res}-{r.last_res}')
    if len(tied_ids) > 1:
        print('Multiple chains tie, so they are likely identical copies of this protein.')

print('\nAlignment for best chain', top.chain, '(top = chain, bottom = query):')
print(alns[top.chain])

No confident match (best: chain A, identity 36.5%, coverage 51.4%).
Check the sequence, lower MIN_IDENTITY/MIN_COVERAGE, or note that the protein may be absent from this file.

Alignment for best chain A (top = chain, bottom = query):
target            2 VKNICVCDFTDKLNFLPLEKTKILCELKPQ----------YGEDIKIIA--NKEYEI---
                  0 .||---..|||-------|||..|...|..----------|..|.|...--|..|||---
query            90 IKN---SQFTD-------EKTLELYLSKEENNKSYLMLTFYLGDLKLMIGHNSPYEISLI

target           47 -----------NCMNNAKVFCPLKDTFINNT-NIKLYSPKLHFEIKDITHKGKNAALYYL
                 60 -----------||.||......||.|...||-..........|..------||.......
query           140 LNVATVNKNMSNCKNNSYNIVLLKTTDVFNTSDLEILEGPIQFSL------GKSSGAFRI

target           95 KIDEEASDIFF-------------SCSIKPKQ-VSGLLEGEVRVNLKKHINEEYSIFNEE
                120 ....-----||-------------|..|||..-....||.--..|......|....|..|
query           194 NVTN-----FFLNNTWKAFTKDKISFLIKPEENCYTILEN--KINKPQLLIEKKTTFYSE

target          141 EDVHVCDFSKGNL

In [ ]:
# chain A = CSS
